In [2]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-anthonastycourse2')

#Chicago
point = ee.Geometry.Point([-87.6298, 41.8781])

### Acquire Landsat data
I defined the study area as a rectangular region around Chicago (coordinates: -88.0, 41.6, -87.5, 42.1) and acquired Landsat 8 Collection 2 Level 2 data for the year 2023. The collection was filtered by cloud cover to select the clearest available image with the least cloudy image from 2023.

In [13]:

roi = ee.Geometry.Rectangle([-88.0, 41.6, -87.5, 42.1])

# Get Landsat 8
l8_collection = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
    .filterBounds(roi) \
    .filterDate('2023-01-01', '2023-12-31') \
    .sort('CLOUD_COVER') \
    .first()

### Image processing
The raw Landsat data required several preprocessing steps. First, I applied scale factors to convert the digital numbers to physical values (reflectance). Then, I selected the optical bands and renamed them to more intuitive names (blue, green, red, nir, swir1, swir2) for easier reference. Finally, I clipped the image to the study area to focus processing on the region of interest. These steps ensured the data was properly calibrated and formatted for analysis.

In [14]:
# Apply scale factors
def applyScaleFactors(image):
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    return image.addBands(opticalBands, None, True).addBands(thermalBands, None, True)

l8_scaled = applyScaleFactors(l8_collection)

optical_bands = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
band_names = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']

l8_renamed = l8_scaled.select(optical_bands).rename(band_names)

# Clip image
clipped_image = l8_renamed.clip(roi)

vis_params = {
    "bands": ['red', 'green', 'blue'],
    "min": 0,
    "max": 0.3
}

Map = geemap.Map()
Map.centerObject(roi, 9)
Map.addLayer(clipped_image, vis_params, "Landsat 8 RGB")
Map.addLayer(roi, {"color": "red"}, "ROI Boundary", False)
display(Map)

Map(center=[41.84994537494724, -87.7500000000001], controls=(WidgetControl(options=['position', 'transparent_b…

### Feature Engineering
To improve classification accuracy, I created multiple derivative features from the original bands:

Spectral indices (NDVI, NDBI, MNDWI) to highlight vegetation, built-up areas, and water bodies
Elevation and slope data from SRTM DEM to capture topographic influences
Texture features using Sobel edge detection to identify boundaries between different land cover types

These additional features provide complementary information that helps distinguish between land cover classes that might be spectrally similar.

In [5]:
# Calculate
ndvi = clipped_image.normalizedDifference(["nir", "red"]).rename("NDVI")
ndbi = clipped_image.normalizedDifference(["swir1", "nir"]).rename("NDBI")
mndwi = clipped_image.normalizedDifference(["green", "swir1"]).rename("MNDWI")

dem = ee.Image("USGS/SRTMGL1_003").clip(roi)
slope = ee.Terrain.slope(dem).rename("slope")
dem = dem.rename("elevation")

clipped_image = clipped_image.addBands([ndvi, ndbi, mndwi, dem, slope])

# Apply Sobel convolution
sobel_kernel = ee.Kernel.sobel()
sobel_blue = clipped_image.select("blue").convolve(sobel_kernel).rename("sobel_blue")
sobel_green = clipped_image.select("green").convolve(sobel_kernel).rename("sobel_green")
sobel_red = clipped_image.select("red").convolve(sobel_kernel).rename("sobel_red")
sobel_nir = clipped_image.select("nir").convolve(sobel_kernel).rename("sobel_nir")

# Add Sobel bands
clipped_image = clipped_image.addBands([sobel_blue, sobel_green, sobel_red, sobel_nir])

print("Available bands:", clipped_image.bandNames().getInfo())

Available bands: ['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'NDVI', 'NDBI', 'MNDWI', 'elevation', 'slope', 'sobel_blue', 'sobel_green', 'sobel_red', 'sobel_nir']


### Data normalization
I normalized all features to a 0-1 range to ensure that no single feature would dominate the classification due to different value ranges. This improves classifier performance by giving each feature equal consideration during the training process.

In [6]:
# Normalize
band_names = ["blue", "green", "red", "nir", "swir1", "swir2",
              "NDVI", "NDBI", "MNDWI", "elevation", "slope",
              "sobel_blue", "sobel_green", "sobel_red", "sobel_nir"]

# Normalization function
def normalize_band(image, band):
    min_val = image.select(band).reduceRegion(
        reducer=ee.Reducer.min(),
        geometry=roi,
        scale=30,
        bestEffort=True
    ).getNumber(band)

    max_val = image.select(band).reduceRegion(
        reducer=ee.Reducer.max(),
        geometry=roi,
        scale=30,
        bestEffort=True
    ).getNumber(band)

    return image.expression(
        "(b - min) / (max - min)",
        {"b": image.select(band), "min": min_val, "max": max_val}
    ).rename(band + "_norm")

normalized_bands = [normalize_band(clipped_image, band) for band in band_names]

# Combine normalized bands
normalized_clipped_image = ee.Image.cat(normalized_bands)
print("Normalized bands:", normalized_clipped_image.bandNames().getInfo())

Normalized bands: ['blue_norm', 'green_norm', 'red_norm', 'nir_norm', 'swir1_norm', 'swir2_norm', 'NDVI_norm', 'NDBI_norm', 'MNDWI_norm', 'elevation_norm', 'slope_norm', 'sobel_blue_norm', 'sobel_green_norm', 'sobel_red_norm', 'sobel_nir_norm']


### Training data preparation
I processed the manually labeled GeoJSON file containing training points for four land cover classes: urban (0), bare land (1), water (2), and vegetation (3). The processing involved converting MultiPoint features to individual points and assigning the appropriate class values. The final training dataset contained 496 points with a relatively balanced distribution across classes (urban: 138, bare: 124, water: 109, vegetation: 125).

In [7]:
import json
from google.colab import drive

# Mount Google Drive
try:
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully")
except:
    print("Google Drive already mounted or mount failed")

geojson_path = '/content/drive/MyDrive/landcover_training_points.geojson'

try:

    with open(geojson_path, 'r') as f:
        geojson_data = json.load(f)

    print(f"GeoJSON type: {geojson_data.get('type', 'Unknown')}")
    print(f"Original feature count: {len(geojson_data.get('features', []))}")


    new_features = []
    class_map = {"urban": 0, "bare": 1, "water": 2, "vegetation": 3}

    for feature_idx, feature in enumerate(geojson_data.get('features', [])):
        geom = feature.get('geometry', {})
        props = feature.get('properties', {})

        # If MultiPoint type, split into separate Points
        if geom.get('type') == 'MultiPoint':
            coords = geom.get('coordinates', [])


            class_value = None
            if feature_idx == 0:
                class_value = 0  # urban
            elif feature_idx == 1:
                class_value = 1  # bare
            elif feature_idx == 2:
                class_value = 2  # water
            elif feature_idx == 3:
                class_value = 3  # vegetation


            for coord_idx, coord in enumerate(coords):
                new_feature = {
                    "type": "Feature",
                    "geometry": {
                        "type": "Point",
                        "coordinates": coord
                    },
                    "properties": {
                        "class": class_value,
                        "original_feature": feature_idx,
                        "point_index": coord_idx
                    }
                }
                new_features.append(new_feature)


        elif geom.get('type') == 'Point':
            if feature_idx == 0:
                class_value = 0
            elif feature_idx == 1:
                class_value = 1
            elif feature_idx == 2:
                class_value = 2
            elif feature_idx == 3:
                class_value = 3

            props['class'] = class_value
            feature['properties'] = props
            new_features.append(feature)

    fixed_geojson = {
        "type": "FeatureCollection",
        "features": new_features
    }

    fixed_path = '/content/fixed_training_points.geojson'
    with open(fixed_path, 'w') as f:
        json.dump(fixed_geojson, f)

    print(f"Fixed GeoJSON saved to: {fixed_path}")
    print(f"Fixed feature count: {len(new_features)}")

    # Count points per class
    class_counts = {0: 0, 1: 0, 2: 0, 3: 0}
    for feature in new_features:
        class_value = feature['properties']['class']
        class_counts[class_value] = class_counts.get(class_value, 0) + 1

    # Display class distribution
    class_names = ['urban', 'bare', 'water', 'vegetation']
    print("\nFixed class distribution:")
    for i, name in enumerate(class_names):
        print(f"  Class {i} ({name}): {class_counts.get(i, 0)} points")

    training_points = geemap.geojson_to_ee(fixed_path)

    training_map = geemap.Map()
    training_map.centerObject(roi, 12)
    training_map.addLayer(normalized_clipped_image,
                         {"bands": ["red_norm", "green_norm", "blue_norm"],
                          "min": 0, "max": 1},
                         "Normalized Image")

    colors = ['red', 'yellow', 'blue', 'green']
    for i, name in enumerate(class_names):
        class_points = training_points.filter(ee.Filter.eq('class', i))
        training_map.addLayer(class_points, {'color': colors[i]}, f'{name} points')

    training_map.add_layer_manager()
    display(training_map)

except Exception as e:
    print(f"Error processing GeoJSON file: {e}")

Mounted at /content/drive
Google Drive mounted successfully
GeoJSON type: FeatureCollection
Original feature count: 4
Fixed GeoJSON saved to: /content/fixed_training_points.geojson
Fixed feature count: 496

Fixed class distribution:
  Class 0 (urban): 138 points
  Class 1 (bare): 124 points
  Class 2 (water): 109 points
  Class 3 (vegetation): 125 points


Map(center=[41.84994537494724, -87.7500000000001], controls=(WidgetControl(options=['position', 'transparent_b…

### Sampling and data splitting
I extracted feature values at each training point location and split the data into training (70%) and testing (30%) sets. This resulted in 345 training samples and 147 testing samples, with all classes well-represented in both sets. This split allows for both robust model training and independent accuracy assessment.

In [8]:
# Sample the normalized image at training points
samples = normalized_clipped_image.sampleRegions(
    collection=training_points,
    properties=["class"],
    scale=30
)

print("Sample count:", samples.size().getInfo())

# Split into training and testing sets
training_data = samples.randomColumn("random", seed=123)
training_samples = training_data.filter(ee.Filter.lt("random", 0.7))
testing_samples = training_data.filter(ee.Filter.gte("random", 0.7))

print("Training samples:", training_samples.size().getInfo())
print("Testing samples:", testing_samples.size().getInfo())

#Also per class
train_class_counts = training_samples.aggregate_histogram('class').getInfo()
test_class_counts = testing_samples.aggregate_histogram('class').getInfo()

print("Training samples per class:", train_class_counts)
print("Testing samples per class:", test_class_counts)

Sample count: 492
Training samples: 345
Testing samples: 147
Training samples per class: {'0': 98, '1': 81, '2': 69, '3': 97}
Testing samples per class: {'0': 40, '1': 43, '2': 37, '3': 27}


### Classification and Evaluation

I tried three different Random Forest classifiers, each with their own twist.

My basic classifier with just 10 trees hit 84.35% accuracy using standard spectral bands and indices. Not too shabby for a starter model! But when I added texture features (the Sobel filters), accuracy jumped to 86.39% without changing anything else. That extra 2% might not sound like much, but in remote sensing, small improvements like this can make a real difference.

Interestingly, my "improved" model with 50 trees and parameter tuning actually performed slightly worse than the convolution model (85.71%). Sometimes more complexity doesn't pay off, which is a good reminder about the risk of overfitting.

Looking at individual classes, bare land was practically a slam dunk with 100% producer's accuracy in my best model - meaning we caught every bare land pixel correctly. Water was incredibly reliable too, with 96.67% user's accuracy - so when the model says "water," you can pretty much trust it.

Urban areas gave me the most trouble, with accuracy in the mid-70s. Anyone who's worked with urban remote sensing knows why - cities are a complex mix of buildings, roads, small green spaces, and shadows, creating a challenging spectral puzzle.

When I checked what features mattered most, NDVI (vegetation index) dominated at 17.75, followed closely by NDBI (built-up index) at 17.00. My texture features ranked at the bottom of the importance list, which was puzzling considering they improved overall accuracy. It reminds me that feature importance metrics don't always tell the whole story about how information combines in complex ways within the model.

In [9]:
classifier = ee.Classifier.smileRandomForest(numberOfTrees=10)

# Train classifier 1 (without kernel filter features)
trained_classifier_1 = classifier.train(
    features=training_samples,
    classProperty="class",
    inputProperties=[
        "blue_norm", "green_norm", "red_norm", "nir_norm", "swir1_norm",
        "swir2_norm", "NDVI_norm", "NDBI_norm", "MNDWI_norm",
        "elevation_norm", "slope_norm"
    ]
)

# Train classifier 2 (with kernel filter features)
trained_classifier_2 = classifier.train(
    features=training_samples,
    classProperty="class",
    inputProperties=[
        "blue_norm", "green_norm", "red_norm", "nir_norm", "swir1_norm",
        "swir2_norm", "NDVI_norm", "NDBI_norm", "MNDWI_norm",
        "elevation_norm", "slope_norm", "sobel_blue_norm", "sobel_green_norm",
        "sobel_red_norm", "sobel_nir_norm"
    ]
)

# Evaluate classifier 1
classified_testing_samples_1 = testing_samples.classify(trained_classifier_1)
confusion_matrix_1 = classified_testing_samples_1.errorMatrix("class", "classification")

# Classifier 1 results
print("Classifier 1 (Without Convolution) Results:")
print("Confusion Matrix:")
print(confusion_matrix_1.getInfo())
print("Label Order:", confusion_matrix_1.order().getInfo())
print(f"Overall Accuracy: {confusion_matrix_1.accuracy().getInfo():.4f}")
print("User's Accuracy (Precision):", confusion_matrix_1.consumersAccuracy().getInfo())
print("Producer's Accuracy (Recall):", confusion_matrix_1.producersAccuracy().getInfo())

# Evaluate classifier 2
classified_testing_samples_2 = testing_samples.classify(trained_classifier_2)
confusion_matrix_2 = classified_testing_samples_2.errorMatrix("class", "classification")

# Classifier 2 results
print("\nClassifier 2 (With Convolution) Results:")
print("Confusion Matrix:")
print(confusion_matrix_2.getInfo())
print("Label Order:", confusion_matrix_2.order().getInfo())
print(f"Overall Accuracy: {confusion_matrix_2.accuracy().getInfo():.4f}")
print("User's Accuracy (Precision):", confusion_matrix_2.consumersAccuracy().getInfo())
print("Producer's Accuracy (Recall):", confusion_matrix_2.producersAccuracy().getInfo())

# Train improved classifier (more trees)
improved_classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=50,
    minLeafPopulation=3,
    bagFraction=0.7
)

improved_trained = improved_classifier.train(
    features=training_samples,
    classProperty="class",
    inputProperties=[
        "blue_norm", "green_norm", "red_norm", "nir_norm", "swir1_norm",
        "swir2_norm", "NDVI_norm", "NDBI_norm", "MNDWI_norm",
        "elevation_norm", "slope_norm", "sobel_blue_norm", "sobel_green_norm",
        "sobel_red_norm", "sobel_nir_norm"
    ]
)

# Evaluate improved classifier
classified_testing_samples_3 = testing_samples.classify(improved_trained)
confusion_matrix_3 = classified_testing_samples_3.errorMatrix("class", "classification")

# Improved classifier results
print("\nImproved Classifier Results:")
print("Confusion Matrix:")
print(confusion_matrix_3.getInfo())
print("Label Order:", confusion_matrix_3.order().getInfo())
print(f"Overall Accuracy: {confusion_matrix_3.accuracy().getInfo():.4f}")
print("User's Accuracy (Precision):", confusion_matrix_3.consumersAccuracy().getInfo())
print("Producer's Accuracy (Recall):", confusion_matrix_3.producersAccuracy().getInfo())

# Feature importance
importance = improved_trained.explain().get("importance")
importance_dict = importance.getInfo()

sorted_features = sorted(importance_dict.items(), key=lambda item: item[1], reverse=True)

print("\nFeature Importance:")
for feature, importance_value in sorted_features:
    print(f"{feature}: {importance_value}")

Classifier 1 (Without Convolution) Results:
Confusion Matrix:
[[29, 0, 1, 10], [1, 42, 0, 0], [1, 5, 30, 1], [4, 0, 0, 23]]
Label Order: [0, 1, 2, 3]
Overall Accuracy: 0.8435
User's Accuracy (Precision): [[0.8285714285714286, 0.8936170212765957, 0.967741935483871, 0.6764705882352942]]
Producer's Accuracy (Recall): [[0.725], [0.9767441860465116], [0.8108108108108109], [0.8518518518518519]]

Classifier 2 (With Convolution) Results:
Confusion Matrix:
[[30, 0, 1, 9], [0, 43, 0, 0], [3, 5, 29, 0], [2, 0, 0, 25]]
Label Order: [0, 1, 2, 3]
Overall Accuracy: 0.8639
User's Accuracy (Precision): [[0.8571428571428571, 0.8958333333333334, 0.9666666666666667, 0.7352941176470589]]
Producer's Accuracy (Recall): [[0.75], [1], [0.7837837837837838], [0.9259259259259259]]

Improved Classifier Results:
Confusion Matrix:
[[31, 0, 0, 9], [1, 42, 0, 0], [3, 5, 29, 0], [3, 0, 0, 24]]
Label Order: [0, 1, 2, 3]
Overall Accuracy: 0.8571
User's Accuracy (Precision): [[0.8157894736842105, 0.8936170212765957, 1, 0.

Apply the best classifier to the image
I applied the best-performing classifier (convolution-based) to the entire image, resulting in a classified land cover map. The visualization used a color scheme of red (urban), yellow (bare), blue (water), and green (vegetation) to clearly distinguish between classes.

In [10]:
# Use the classifier with highest accuracy
best_classifier = trained_classifier_2

# Apply to the entire image
classified_image = normalized_clipped_image.classify(best_classifier)

# Result
class_vis = {
    "min": 0,
    "max": 3,
    "palette": ['red', 'yellow', 'blue', 'green']
}

class_map = geemap.Map()
class_map.centerObject(roi, 12)
class_map.addLayer(clipped_image, vis_params, "Landsat RGB", False)
class_map.addLayer(classified_image, class_vis, "Land Cover Classification")
class_map.add_layer_manager()
display(class_map)

Map(center=[41.84994537494724, -87.7500000000001], controls=(WidgetControl(options=['position', 'transparent_b…

In [11]:
# Load ESA WorldCover data
esa_worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .filterBounds(roi)
    .first()
)

clipped_worldcover = esa_worldcover.clip(roi)

# Define color mapping for ESA data
color_map = {
    10: '#006400',  # Tree Cover
    20: '#FFBB22',  # Shrubland
    30: '#FFFF4C',  # Grassland
    40: '#F096FF',  # Cropland
    50: '#FA0000',  # Built-up
    60: '#B4B4B4',  # Bare
    70: '#F0F0F0',  # Snow/Ice
    80: '#0064C8',  # Water
    90: '#0096A0',  # Wetland
}

# Convert dictionary to color palette
palette = [color_map[key] for key in sorted(color_map.keys())]

# Define visualization parameters
vis_params_esa = {
    'min': 10,
    'max': 90,
    'palette': palette
}

# Remap ESA classes to match
reclassified_esa = clipped_worldcover.remap(
    [10, 20, 30, 40, 50, 60, 70, 80, 90],
    [3, 3, 3, 3, 0, 1, 1, 2, 2]
)

# Calculate agreement
agreement = classified_image.eq(reclassified_esa)
agreement_percent = agreement.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=roi,
    scale=30,
    maxPixels=1e9
).get('classification')

print("Agreement with ESA data:", ee.Number(agreement_percent).multiply(100).getInfo(), "%")

compare_map = geemap.Map()
compare_map.centerObject(roi, 12)

compare_map.addLayer(classified_image, class_vis, "Your Classification")

compare_map.addLayer(clipped_worldcover, vis_params_esa, "ESA WorldCover Original", False)

compare_map.addLayer(reclassified_esa, class_vis, "Reclassified ESA", False)

compare_map.addLayer(agreement.selfMask(), {"palette": ['white']}, "Agreement Areas", False)

compare_map.add_layer_manager()
display(compare_map)

Agreement with ESA data: 44.884364385147656 %


Map(center=[41.84994537494724, -87.7500000000001], controls=(WidgetControl(options=['position', 'transparent_b…

In [16]:
export_task = ee.batch.Export.image.toDrive(
    image=classified_image,
    description="LandCoverClassification",
    scale=30,
    region=roi.getInfo()["coordinates"],
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)

export_task.start()

In [15]:
import pandas as pd

# Confusion matrix dataframe
conf_matrix_df = pd.DataFrame(
    confusion_matrix_2.getInfo(),
    columns=["Predicted Urban", "Predicted Bare", "Predicted Water", "Predicted Vegetation"],
    index=["Actual Urban", "Actual Bare", "Actual Water", "Actual Vegetation"]
)

# Accuracy metrics dataframe
accuracy_df = pd.DataFrame({
    'Class': ['Urban', 'Bare', 'Water', 'Vegetation', 'Overall'],
    'Producer Accuracy': [0.75, 1.0, 0.7838, 0.9259, 0.8639],
    'User Accuracy': [0.8571, 0.8958, 0.9667, 0.7353, 0.8639]
})

conf_matrix_df.to_csv('confusion_matrix.csv')
accuracy_df.to_csv('accuracy_metrics.csv')

# Comparing My Classification with ESA WorldCover Data

When I compared my classification with ESA's WorldCover data, I found a 44.88% agreement, which initially seemed low but makes sense given the fundamental differences in our approaches.

Looking closer at the differences, I noticed several key factors affecting the comparison:

First, ESA uses Sentinel-1 and Sentinel-2 imagery (10-20m resolution with more spectral bands), while I used Landsat 8 (30m resolution). It's like comparing photos taken with different cameras – the same landscape can look quite different.

Second, ESA's classification scheme is much more detailed than mine. Their 10-class system had to be simplified to match my 4 classes, creating inevitable mapping conflicts. For example, ESA distinguishes between forest, shrubland, and grassland, while I grouped all as "vegetation" – creating disagreement in mixed vegetation areas like parks and residential neighborhoods with tree cover.

I also noticed several specific features my model may struggle with:

1. **Urban-rural transitions**: My classification tended to extend urban boundaries further into vegetated areas than ESA did, particularly in suburban zones.

2. **Wetlands and shallow water**: These were challenging for my model, as wetlands have unique spectral properties between water and vegetation that Landsat's resolution struggles to capture consistently.

3. **Road networks**: ESA's data seemed to better capture major transportation corridors as part of the urban class, likely due to Sentinel's higher resolution. Landsat's 30m pixels often miss narrow linear features like roads.

4. **Building shadows**: In dense urban areas, my model occasionally confused dark shadows from tall buildings with water bodies.

# Reflection


**1.   Limitations and Future Improvements**

The biggest headache was dealing with band naming in Google Earth Engine - one wrong name and everything breaks. I spent way too much time debugging silly errors like trying to use "SR_B3" when the system only recognized "green." Next time, I'd be much more careful about keeping track of these names throughout the workflow.

If I had more time, I'd love to try a multi-seasonal approach. Chicago vegetation looks completely different in summer versus winter, so using images from different seasons might help distinguish urban areas from dormant vegetation. I also wish I'd had time to implement proper cross-validation instead of a simple train-test split. It would have given me more confidence in the accuracy numbers.

**2.   Impact of Feature Engineering**

Adding the Sobel filters boosted accuracy from 84.35% to 86.39% without changing anything else. That said, when I looked at feature importance, the indices (NDVI, NDBI, MNDWI) dominated everything else. NDVI alone contributed 17.75 of the importance weight, which makes sense - it's the gold standard for vegetation mapping for a reason.

What really threw me was seeing the texture features rank at the bottom of importance, despite improving overall accuracy. I think what's happening is that they're helping resolve specific difficult cases at class boundaries, even though they're not broadly important across the whole image. It's a good reminder that feature importance isn't everything.

**3. Training Data Creation Challenges**

It took forever to carefully pick points that truly represented each class. The hardest part was finding "pure" urban or vegetation areas - there's so much mixing at the urban-suburban interface. I'd look at what seemed like a clear urban block, zoom in, and suddenly spot trees and small green spaces everywhere.
My class distribution wasn't terrible (urban: 138, bare: 124, water: 109, vegetation: 125), but it wasn't perfect either. In hindsight, I should have implemented stratified sampling to ensure exactly equal representation. One approach I've read about but haven't tried is "active learning," where the algorithm itself suggests which areas need more training points based on uncertainty. That would be amazing to try if I do this again.

**4. Class-Specific Performance Differences**

My model absolutely crushed it with bare land (100% producer's accuracy) but struggled more with urban areas (75% producer's accuracy). This makes total sense - bare land has a pretty distinctive spectral signature, while urban areas are a complex mess of different materials, shadows, and mixed pixels.
The practical implications of this uneven performance really depend on the project goals. If I were mapping impervious surfaces for stormwater management, the urban accuracy would be my biggest concern. But if I were tracking desertification or mining activities, the high bare land accuracy would be exactly what I needed.

What's interesting is that water had high user's accuracy (96.67%) but lower producer's accuracy (78.38%), meaning the model missed some water pixels but rarely mislabeled anything as water. For something like flood mapping, that conservative approach might actually be preferable to false positives.

> The code debugging and ESA Comparison part in this project were assisted by ChatGPT.
>
> Source: OpenAI ChatGPT, accessed on April 23rd, 2025.
